In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# train_v7_rtmdet_cspnext_lss_bev_cleaned_colab

Colab-friendly copy of the `v7` training notebook.

It is adapted for:
- kaggle-safe dataset folders where `:` in filenames was replaced during repack;
- training or resuming from uploaded checkpoints on Colab;
- continuing either the RTMDet/CSPNeXt run or the appended ConvNeXt V2 FCMAE-based run.


In [ ]:
from src.utils.dist import barrier, cleanup_distributed, get_local_rank, get_rank, get_world_size, is_dist_enabled, is_main_process, setup_distributed

%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Adjust these paths to wherever Colab/KaggleHub placed your dataset and uploaded artifacts.
DATA_TRAIN = Path('/content/autonomy_yandex_dataset_train_kaggle_safe')
DATA_VAL   = Path('/content/autonomy_yandex_dataset_val_kaggle_safe')
DATA_TEST  = Path('/content/autonomy_yandex_dataset_test_kaggle_safe')
ARTIFACTS_DIR = Path('/content/model_artifacts')
RTMDET_PRETRAIN_PATH = ARTIFACTS_DIR / 'rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth'

cfg = {
    'run_dir': '/content/runs/v7_rtmdet_cspnext_lss_bev_cleaned',
    'img_hw': (384, 704),
    'batch_size': 1,
    'val_batch_size': 1,
    'grad_accum': 16,
    'epochs': 24,
    'warmup_epochs': 2,
    'freeze_backbone_epochs': 2,
    'unfreeze_last_stages': 2,
    'lr_backbone': 2e-5,
    'lr_image_neck': 1e-4,
    'lr_lss_bev': 1.5e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 2,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'rtmdet_backbone_ckpt': str(RTMDET_PRETRAIN_PATH),
    'csp_arch': 'P5',
    'csp_deepen_factor': 1.0,
    'csp_widen_factor': 1.0,
    'csp_expand_ratio': 0.5,
    'csp_channel_attention': True,
    'csp_out_indices': (2, 3, 4),
    'fpn_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'resume_training': True,
    'resume_ckpt': str(ARTIFACTS_DIR / 'v7_last.pt'),
    'use_ddp': False,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('cuda devices visible:', torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            try:
                print(i, torch.cuda.get_device_name(i))
            except Exception as e:
                print(i, f'<cuda init error: {e}>')
    print('world_size:', get_world_size(), '| rank:', get_rank(), '| local_rank:', get_local_rank())
    print('img_hw:', cfg['img_hw'])
    print('train batch_size(per gpu):', cfg['batch_size'], '| train workers:', cfg['num_workers'])
    print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])
    print('rtmdet_backbone_ckpt:', cfg['rtmdet_backbone_ckpt'])
    print('rtmdet checkpoint exists:', RTMDET_PRETRAIN_PATH.exists())
    print('resume_training:', cfg['resume_training'], '| resume_ckpt exists:', Path(cfg['resume_ckpt']).exists())

if is_main_process():
    with open(RUN_DIR / 'config.json', 'w') as f:
        json.dump(json.loads(json.dumps(cfg, default=str)), f, indent=2)


In [ ]:
from src.geometry import kaggle_safe_name, load_info_with_root, remap_kaggle_paths, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
train_info = remap_kaggle_paths(train_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(val_info, DATA_TRAIN, DATA_VAL, DATA_TEST)

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))

if is_main_process() and len(val_info):
    row = val_info.iloc[0]
    for key in [CAMERA_NAMES[0], INTRINSICS_NAMES[0], CAR2CAM_NAMES[0], GT_NAME]:
        path = resolve_info_path(Path(row['__data_root']), row[key])
        print(key, path, path.exists())


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.cspnext import CSPLayer, CSPNeXtBackboneFromRTMDet, CSPNeXtBlock, ChannelAttention, ConvBNAct, SPPBottleneck, _RTMDetMultiScaleBackbone, load_rtmdet_pretrained_backbone
from src.models.strong_v4 import StrongBEVEncoderDecoder
from src.models.v7 import MultiCamBEVv7RTMDetCSPNeXtLSSClean
from src.utils.training import extract_state_dict, load_resume_state
from src.models.lss import ASPP2d, ConvGNAct, LSSViewTransform2D, ResidualBlock2d, gn_groups



In [ ]:
from src.eval import evaluate_iou
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch
from src.utils.training import cleanup_cuda, unwrap_model



In [ ]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

train_sampler_warm = None
train_sampler_aug = None
if is_dist_enabled():
    train_sampler_warm = DistributedSampler(ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    train_sampler_aug = DistributedSampler(ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_warm is None),
    sampler=train_sampler_warm,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_aug is None),
    sampler=train_sampler_aug,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = None
if is_main_process():
    loader_val = DataLoader(
        ds_val,
        batch_size=cfg['val_batch_size'],
        shuffle=False,
        num_workers=cfg['val_num_workers'],
        pin_memory=(device.type == 'cuda'),
    )

if is_main_process():
    print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
    print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
    if loader_val is not None:
        print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

    sample = ds_train_warm[0]
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            print(k, tuple(v.shape), v.dtype)
        else:
            print(k, type(v), v)


In [ ]:
from src.utils.training import freeze_all_backbone, update_ema

base_model = MultiCamBEVv7RTMDetCSPNeXtLSSClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    pretrained_backbone_path=cfg['rtmdet_backbone_ckpt'],
    csp_arch=cfg['csp_arch'],
    csp_deepen_factor=cfg['csp_deepen_factor'],
    csp_widen_factor=cfg['csp_widen_factor'],
    csp_expand_ratio=cfg['csp_expand_ratio'],
    csp_channel_attention=cfg['csp_channel_attention'],
    csp_out_indices=cfg['csp_out_indices'],
    fpn_dim=cfg['fpn_dim'],
    context_dim=cfg['context_dim'],
    depth_bins=cfg['depth_bins'],
    depth_min=cfg['depth_min'],
    depth_max=cfg['depth_max'],
    world_z_min=cfg['world_z_min'],
    world_z_max=cfg['world_z_max'],
    bev_base_channels=cfg['bev_base_channels'],
    bev_gn_groups=cfg['bev_gn_groups'],
).to(device)

def unfreeze_last_stages(model, n_last_stages=2):
    unwrap_model(model).backbone.unfreeze_last_stages(n_last_stages)

freeze_all_backbone(base_model)

if is_dist_enabled():
    model = DDP(
        base_model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        broadcast_buffers=False,
        find_unused_parameters=False,
    )
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params = []
embed_params = []
image_neck_params = []
other_params = []
for name, p in core_model.named_parameters():
    if name.startswith('backbone.backbone.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.smooth') or name.startswith('backbone.out_proj.'):
        image_neck_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': image_neck_params, 'lr': cfg['lr_image_neck'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and cfg['use_amp']))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

resume_state = load_resume_state(core_model, ema_model, optimizer, scheduler, scaler, RUN_DIR)
if resume_state['start_epoch'] >= cfg['freeze_backbone_epochs']:
    unfreeze_last_stages(model, n_last_stages=cfg['unfreeze_last_stages'])

if is_main_process():
    total_params = sum(p.numel() for p in core_model.parameters())
    trainable_params = sum(p.numel() for p in core_model.parameters() if p.requires_grad)
    backbone_total = sum(p.numel() for name, p in core_model.named_parameters() if name.startswith('backbone.backbone.'))
    neck_total = sum(p.numel() for name, p in core_model.named_parameters() if name.startswith('backbone.laterals.') or name.startswith('backbone.smooth') or name.startswith('backbone.out_proj.'))
    lss_bev_total = sum(p.numel() for name, p in core_model.named_parameters() if not name.startswith('backbone.backbone.') and not name.startswith('backbone.laterals.') and not name.startswith('backbone.smooth') and not name.startswith('backbone.out_proj.') and not name.startswith('rover_'))
    rover_total = sum(p.numel() for name, p in core_model.named_parameters() if name.startswith('rover_'))

    print('params M total:', total_params / 1e6)
    print('params M trainable:', trainable_params / 1e6)
    print('params M backbone:', backbone_total / 1e6)
    print('params M image_neck:', neck_total / 1e6)
    print('params M lss_bev:', lss_bev_total / 1e6)
    print('params M rover:', rover_total / 1e6)
    print('backbone frozen at start:', not any(p.requires_grad for p in core_model.backbone.backbone.parameters()))
    print('resume enabled:', resume_state['enabled'])
    print('resume start_epoch:', resume_state['start_epoch'])
    print('backbone load summary:', core_model.backbone.backbone_load_summary)

    with torch.no_grad():
        batch = next(iter(loader_warm))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = core_model.forward_debug(images, intr, c2c, rover_id)

        print('feat_s8 shape:', tuple(dbg['feat_s8'].shape), 'finite:', torch.isfinite(dbg['feat_s8']).all().item())
        print('feat_s16 shape:', tuple(dbg['feat_s16'].shape), 'finite:', torch.isfinite(dbg['feat_s16']).all().item())
        print('feat_s32 shape:', tuple(dbg['feat_s32'].shape), 'finite:', torch.isfinite(dbg['feat_s32']).all().item())
        print('image_fused shape:', tuple(dbg['image_fused'].shape), 'finite:', torch.isfinite(dbg['image_fused']).all().item())
        print('depth_prob shape:', tuple(dbg['depth_prob'].shape), 'finite:', torch.isfinite(dbg['depth_prob']).all().item())
        print('bev_raw shape:', tuple(dbg['bev_raw'].shape), 'finite:', torch.isfinite(dbg['bev_raw']).all().item())
        print('valid_ratio:', dbg['valid_ratio'])
        print('logits shape:', tuple(dbg['logits'].shape), 'finite:', torch.isfinite(dbg['logits']).all().item())

    core_model.train()
    optimizer.zero_grad(set_to_none=True)
    batch = next(iter(loader_warm))
    images = batch['images'].to(device, non_blocking=True)
    intr = batch['intrinsics'].to(device, non_blocking=True)
    c2c = batch['car2cams'].to(device, non_blocking=True)
    rover_id = batch['rover_id'].to(device, non_blocking=True)
    gt = batch['gt'].to(device, non_blocking=True)

    with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
        logits = model(images, intr, c2c, rover_id)
    logits = logits.float()
    loss, parts = loss_fn(logits, gt)
    print('sanity loss:', float(loss), '| finite:', torch.isfinite(loss).item(), '| bce:', parts['bce'])
    if not torch.isfinite(loss):
        raise RuntimeError('Sanity backward aborted because loss is non-finite')
    scaler.scale(loss).backward()
    optimizer.zero_grad(set_to_none=True)
    core_model.eval()
    ema_model.eval()

cleanup_cuda()
barrier()


In [ ]:
log = list(resume_state['log'])
best_iou = float(resume_state['best_iou'])
best_ema_iou = float(resume_state['best_ema_iou'])
start = time.time() - float(resume_state['elapsed_minutes']) * 60.0
start_epoch = int(resume_state['start_epoch'])
backbone_unfrozen = bool(start_epoch >= cfg['freeze_backbone_epochs'])

for epoch in range(start_epoch, cfg['epochs']):
    if train_sampler_warm is not None:
        train_sampler_warm.set_epoch(epoch)
    if train_sampler_aug is not None:
        train_sampler_aug.set_epoch(epoch)

    if (not backbone_unfrozen) and epoch >= cfg['freeze_backbone_epochs']:
        unfreeze_last_stages(model, n_last_stages=cfg['unfreeze_last_stages'])
        backbone_unfrozen = True
        if is_main_process():
            print(f'unfroze last {cfg["unfreeze_last_stages"]} CSPNeXt stages at epoch {epoch:02d}')

    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]', disable=not is_main_process())
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss detected at epoch={epoch} step={step}: {loss.item()}')

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()
    barrier()

    val_iou_05 = None
    val_iou_05_ema = None
    if is_main_process():
        cleanup_cuda()
        print('start val model @0.5')
        val_iou_05 = evaluate_iou(core_model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

        cleanup_cuda()
        print('start val ema @0.5')
        val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - start) / 60
        backbone_grad_enabled = any(p.requires_grad for p in core_model.backbone.backbone.parameters())

        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'backbone_trainable': bool(backbone_grad_enabled),
            'minutes': float(elapsed),
        }

        print(
            f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
            f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
            f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
        )

        if val_iou_05 > best_iou:
            best_iou = val_iou_05
            torch.save({
                'model_type': 'v7_rtmdet_cspnext_lss_bev_cleaned',
                'model': core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', val_iou_05)

        if val_iou_05_ema > best_ema_iou:
            best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v7_rtmdet_cspnext_lss_bev_cleaned',
                'model': core_model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', val_iou_05_ema)

        log.append(row)
        pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v7_rtmdet_cspnext_lss_bev_cleaned',
            'model': core_model.state_dict(),
            'ema': ema_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict() if scaler is not None else None,
            'epoch': epoch,
            'best_iou': float(best_iou),
            'best_ema_iou': float(best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

    barrier()


### Notes

- This notebook intentionally does **not** do threshold sweep inside training.
- Use a separate inference notebook after training to compare `best.pt` and `ema_best.pt`, pick the best global threshold, and build submissions.
- The only warm start in this notebook is the local RTMDet backbone checkpoint; there is no transfer from the DINO `v6` run.


## Next: ConvNeXt V2-B FCMAE-Based Run

These cells are meant to be executed **after** the RTMDet training finishes in the same notebook session, so we do not waste GPU time.


In [ ]:
import subprocess
import sys

for _name in ['base_model', 'model', 'core_model', 'ema_model', 'optimizer', 'scheduler', 'scaler', 'loss_fn']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
cleanup_cuda()
barrier()

def ensure_package(pkg: str):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure_package('timm')
ensure_package('safetensors')

import timm
from safetensors.torch import load_file as safe_load_file

CONVNEXT_DIR = ARTIFACTS_DIR / 'convnext'
CONVNEXT_CKPT = CONVNEXT_DIR / 'model.safetensors'
CONVNEXT_CONFIG = CONVNEXT_DIR / 'config.json'

convnext_cfg = {
    'run_dir': '/content/runs/v8_convnextv2_fcmae_lss_bev_cleaned',
    'img_hw': (384, 704),
    'batch_size': 1,
    'val_batch_size': 1,
    'grad_accum': 16,
    'epochs': 24,
    'warmup_epochs': 2,
    'freeze_backbone_epochs': 2,
    'unfreeze_last_stages': 2,
    'lr_backbone': 1.5e-5,
    'lr_image_neck': 1.0e-4,
    'lr_lss_bev': 1.5e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'convnext_model_name': 'convnextv2_base.fcmae_ft_in22k_in1k_384',
    'convnext_fallback_model_name': 'convnextv2_base.fcmae_ft_in22k_in1k',
    'convnext_ckpt': str(CONVNEXT_CKPT),
    'fpn_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'resume_training': True,
    'resume_ckpt': str(ARTIFACTS_DIR / 'v8_last.pt'),
    'use_amp': True,
}

CONVNEXT_RUN_DIR = Path(convnext_cfg['run_dir'])
CONVNEXT_RUN_DIR.mkdir(parents=True, exist_ok=True)
(CONVNEXT_RUN_DIR / 'preproc_cache').mkdir(parents=True, exist_ok=True)

if is_main_process():
    with open(CONVNEXT_RUN_DIR / 'config.json', 'w') as f:
        json.dump(json.loads(json.dumps(convnext_cfg, default=str)), f, indent=2)
    print('convnext checkpoint exists:', CONVNEXT_CKPT.exists())
    print('convnext config exists:', CONVNEXT_CONFIG.exists())

convnext_ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=convnext_cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
convnext_ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=convnext_cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
convnext_ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=convnext_cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

convnext_train_sampler_warm = None
convnext_train_sampler_aug = None
if is_dist_enabled():
    convnext_train_sampler_warm = DistributedSampler(convnext_ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    convnext_train_sampler_aug = DistributedSampler(convnext_ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

convnext_loader_warm = DataLoader(
    convnext_ds_train_warm,
    batch_size=convnext_cfg['batch_size'],
    shuffle=(convnext_train_sampler_warm is None),
    sampler=convnext_train_sampler_warm,
    num_workers=convnext_cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
convnext_loader_train = DataLoader(
    convnext_ds_train_aug,
    batch_size=convnext_cfg['batch_size'],
    shuffle=(convnext_train_sampler_aug is None),
    sampler=convnext_train_sampler_aug,
    num_workers=convnext_cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
convnext_loader_val = None
if is_main_process():
    convnext_loader_val = DataLoader(
        convnext_ds_val,
        batch_size=convnext_cfg['val_batch_size'],
        shuffle=False,
        num_workers=convnext_cfg['val_num_workers'],
        pin_memory=(device.type == 'cuda'),
    )
    print('convnext loader_warm batch_size:', convnext_loader_warm.batch_size)
    print('convnext loader_train batch_size:', convnext_loader_train.batch_size)
    if convnext_loader_val is not None:
        print('convnext loader_val batch_size:', convnext_loader_val.batch_size)


In [ ]:
def _strip_prefix_if_present(state_dict, prefix: str):
    out = {}
    hit = False
    for k, v in state_dict.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
            hit = True
    return out if hit else None


def flexible_load_into_module(module: nn.Module, state_dict: dict):
    current = module.state_dict()
    variants = [('raw', state_dict)]
    for prefix in ['model.', 'module.', 'backbone.', 'model.backbone.', 'module.backbone.']:
        candidate = _strip_prefix_if_present(state_dict, prefix)
        if candidate is not None:
            variants.append((f'strip:{prefix}', candidate))

    best_name = None
    best_filtered = None
    best_matched = -1
    for name, cand in variants:
        filtered = {k: v for k, v in cand.items() if k in current and current[k].shape == v.shape}
        if len(filtered) > best_matched:
            best_name = name
            best_filtered = filtered
            best_matched = len(filtered)

    missing, unexpected = module.load_state_dict(best_filtered, strict=False)
    summary = {
        'variant': best_name,
        'matched_keys': len(best_filtered),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }
    return summary, missing, unexpected


class _ConvNeXtV2MultiScaleBackbone(nn.Module):
    def __init__(self, ckpt_path: str, model_name: str, fallback_model_name: str, fpn_dim: int = 128, groups: int = 8):
        super().__init__()
        self.ckpt_path = Path(ckpt_path)
        self.model_name = None
        self.backbone = None
        errors = []
        for candidate_name in [model_name, fallback_model_name]:
            try:
                self.backbone = timm.create_model(candidate_name, pretrained=False)
                self.model_name = candidate_name
                break
            except Exception as e:
                errors.append((candidate_name, repr(e)))
        if self.backbone is None:
            raise RuntimeError(f'Failed to create ConvNeXtV2 backbone: {errors}')

        if self.ckpt_path.suffix == '.safetensors':
            raw_state = safe_load_file(str(self.ckpt_path))
        else:
            raw = torch.load(self.ckpt_path, map_location='cpu')
            raw_state = raw.get('state_dict', raw.get('model', raw)) if isinstance(raw, dict) else raw

        self.load_summary, missing, unexpected = flexible_load_into_module(self.backbone, raw_state)
        self.load_summary['checkpoint'] = str(self.ckpt_path)
        self.load_summary['model_name'] = self.model_name
        if len(missing):
            print('convnext sample missing:', missing[:20])
        if len(unexpected):
            print('convnext sample unexpected:', unexpected[:20])
        print(json.dumps(self.load_summary, indent=2))

        stages = getattr(self.backbone, 'stages', None)
        stem = getattr(self.backbone, 'stem', None)
        if stages is None or stem is None:
            raise RuntimeError('ConvNeXt backbone is expected to expose .stem and .stages')
        if len(stages) < 4:
            raise RuntimeError(f'Expected 4 ConvNeXt stages, got {len(stages)}')

        channels = [256, 512, 1024]
        self.laterals = nn.ModuleList([nn.Conv2d(c, fpn_dim, 1) for c in channels])
        self.smooth16 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.smooth8 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.out_proj = nn.Sequential(
            ConvGNAct(fpn_dim * 3, fpn_dim, k=1, s=1, p=0, groups=groups, act=True),
            ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True),
        )

    def freeze_all_stages(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_stages(self, n_last_stages=2):
        self.freeze_all_stages()
        stages = getattr(self.backbone, 'stages', None)
        if stages is None:
            raise RuntimeError('ConvNeXt backbone has no .stages attribute for staged unfreeze')
        for stage in list(stages)[-n_last_stages:]:
            for p in stage.parameters():
                p.requires_grad = True
        for aux_name in ['norm_pre', 'head', 'norm']:
            aux = getattr(self.backbone, aux_name, None)
            if aux is not None:
                for p in aux.parameters():
                    p.requires_grad = True

    def forward(self, x):
        x = self.backbone.stem(x)
        x = self.backbone.stages[0](x)
        feat_s8 = self.backbone.stages[1](x)
        feat_s16 = self.backbone.stages[2](feat_s8)
        feat_s32 = self.backbone.stages[3](feat_s16)
        lat8 = self.laterals[0](feat_s8)
        lat16 = self.laterals[1](feat_s16)
        p32 = self.laterals[2](feat_s32)
        p16 = self.smooth16(lat16 + F.interpolate(p32, size=lat16.shape[-2:], mode='bilinear', align_corners=False))
        p8 = self.smooth8(lat8 + F.interpolate(p16, size=lat8.shape[-2:], mode='bilinear', align_corners=False))
        p16_up = F.interpolate(p16, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        p32_up = F.interpolate(p32, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.out_proj(torch.cat([p8, p16_up, p32_up], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'fused': fused,
        }


class MultiCamBEVv8ConvNeXtV2LSSClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 ckpt_path: str = './convnext/model.safetensors',
                 model_name: str = 'convnextv2_base.fcmae_ft_in22k_in1k_384',
                 fallback_model_name: str = 'convnextv2_base.fcmae_ft_in22k_in1k',
                 fpn_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _ConvNeXtV2MultiScaleBackbone(
            ckpt_path=ckpt_path,
            model_name=model_name,
            fallback_model_name=fallback_model_name,
            fpn_dim=fpn_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            self.backbone.freeze_all_stages()

        self.view_transform = LSSViewTransform2D(
            in_c=fpn_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=BEV_H,
            bev_w=BEV_W,
            bev_res=BEV_RES,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )
        self.bev_decoder = StrongBEVEncoderDecoder(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
        feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
        feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'image_fused': fused,
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits': logits,
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        return torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)


convnext_base_model = MultiCamBEVv8ConvNeXtV2LSSClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=convnext_cfg['rover_emb_dim'],
    rover_cond_dim=convnext_cfg['rover_cond_dim'],
    freeze_backbone=False,
    ckpt_path=convnext_cfg['convnext_ckpt'],
    model_name=convnext_cfg['convnext_model_name'],
    fallback_model_name=convnext_cfg['convnext_fallback_model_name'],
    fpn_dim=convnext_cfg['fpn_dim'],
    context_dim=convnext_cfg['context_dim'],
    depth_bins=convnext_cfg['depth_bins'],
    depth_min=convnext_cfg['depth_min'],
    depth_max=convnext_cfg['depth_max'],
    world_z_min=convnext_cfg['world_z_min'],
    world_z_max=convnext_cfg['world_z_max'],
    bev_base_channels=convnext_cfg['bev_base_channels'],
    bev_gn_groups=convnext_cfg['bev_gn_groups'],
).to(device)

def convnext_freeze_all_backbone(model):
    unwrap_model(model).backbone.freeze_all_stages()

def convnext_unfreeze_last_stages(model, n_last_stages=2):
    unwrap_model(model).backbone.unfreeze_last_stages(n_last_stages)

convnext_freeze_all_backbone(convnext_base_model)

if is_dist_enabled():
    convnext_model = DDP(
        convnext_base_model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        broadcast_buffers=False,
        find_unused_parameters=False,
    )
else:
    convnext_model = convnext_base_model

convnext_core_model = unwrap_model(convnext_model)
convnext_backbone_params = []
convnext_embed_params = []
convnext_image_neck_params = []
convnext_other_params = []
for name, p in convnext_core_model.named_parameters():
    if name.startswith('backbone.backbone.'):
        convnext_backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        convnext_embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.smooth') or name.startswith('backbone.out_proj.'):
        convnext_image_neck_params.append(p)
    else:
        convnext_other_params.append(p)

convnext_optimizer = torch.optim.AdamW([
    {'params': convnext_backbone_params, 'lr': convnext_cfg['lr_backbone'], 'weight_decay': convnext_cfg['weight_decay']},
    {'params': convnext_image_neck_params, 'lr': convnext_cfg['lr_image_neck'], 'weight_decay': convnext_cfg['weight_decay']},
    {'params': convnext_other_params, 'lr': convnext_cfg['lr_lss_bev'], 'weight_decay': convnext_cfg['weight_decay']},
    {'params': convnext_embed_params, 'lr': convnext_cfg['lr_lss_bev'], 'weight_decay': convnext_cfg['embed_weight_decay']},
])
convnext_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(convnext_optimizer, T_max=convnext_cfg['epochs'], eta_min=1e-6)
convnext_loss_fn = CompoundLossV2(pos_weight=convnext_cfg['pos_weight']).to(device)
convnext_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and convnext_cfg['use_amp']))
convnext_ema_model = copy.deepcopy(convnext_core_model).to(device).eval()
for p in convnext_ema_model.parameters():
    p.requires_grad = False

if is_main_process():
    print('convnext params M total:', sum(p.numel() for p in convnext_core_model.parameters()) / 1e6)
    print('convnext params M trainable:', sum(p.numel() for p in convnext_core_model.parameters() if p.requires_grad) / 1e6)
    print('convnext load summary:', convnext_core_model.backbone.load_summary)
    with torch.no_grad():
        batch = next(iter(convnext_loader_warm))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = convnext_core_model.forward_debug(images, intr, c2c, rover_id)
        print('convnext feat_s8 shape:', tuple(dbg['feat_s8'].shape), 'finite:', torch.isfinite(dbg['feat_s8']).all().item())
        print('convnext feat_s16 shape:', tuple(dbg['feat_s16'].shape), 'finite:', torch.isfinite(dbg['feat_s16']).all().item())
        print('convnext feat_s32 shape:', tuple(dbg['feat_s32'].shape), 'finite:', torch.isfinite(dbg['feat_s32']).all().item())
        print('convnext image_fused shape:', tuple(dbg['image_fused'].shape), 'finite:', torch.isfinite(dbg['image_fused']).all().item())
        print('convnext depth_prob shape:', tuple(dbg['depth_prob'].shape), 'finite:', torch.isfinite(dbg['depth_prob']).all().item())
        print('convnext bev_raw shape:', tuple(dbg['bev_raw'].shape), 'finite:', torch.isfinite(dbg['bev_raw']).all().item())
        print('convnext valid_ratio:', dbg['valid_ratio'])
        print('convnext logits shape:', tuple(dbg['logits'].shape), 'finite:', torch.isfinite(dbg['logits']).all().item())

    convnext_core_model.train()
    convnext_optimizer.zero_grad(set_to_none=True)
    batch = next(iter(convnext_loader_warm))
    images = batch['images'].to(device, non_blocking=True)
    intr = batch['intrinsics'].to(device, non_blocking=True)
    c2c = batch['car2cams'].to(device, non_blocking=True)
    rover_id = batch['rover_id'].to(device, non_blocking=True)
    gt = batch['gt'].to(device, non_blocking=True)
    with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and convnext_cfg['use_amp'])):
        logits = convnext_model(images, intr, c2c, rover_id)
    logits = logits.float()
    loss, parts = convnext_loss_fn(logits, gt)
    print('convnext sanity loss:', float(loss), '| finite:', torch.isfinite(loss).item(), '| bce:', parts['bce'])
    if not torch.isfinite(loss):
        raise RuntimeError('ConvNeXt sanity backward aborted because loss is non-finite')
    convnext_scaler.scale(loss).backward()
    convnext_optimizer.zero_grad(set_to_none=True)
    convnext_core_model.eval()
    convnext_ema_model.eval()

cleanup_cuda()
barrier()


In [ ]:
convnext_log = []
convnext_best_iou = -1.0
convnext_best_ema_iou = -1.0
convnext_start = time.time()
convnext_backbone_unfrozen = False

for epoch in range(convnext_cfg['epochs']):
    if convnext_train_sampler_warm is not None:
        convnext_train_sampler_warm.set_epoch(epoch)
    if convnext_train_sampler_aug is not None:
        convnext_train_sampler_aug.set_epoch(epoch)

    if (not convnext_backbone_unfrozen) and epoch >= convnext_cfg['freeze_backbone_epochs']:
        convnext_unfreeze_last_stages(convnext_model, n_last_stages=convnext_cfg['unfreeze_last_stages'])
        convnext_backbone_unfrozen = True
        if is_main_process():
            print(f'unfroze last {convnext_cfg["unfreeze_last_stages"]} ConvNeXt stages at epoch {epoch:02d}')

    convnext_model.train()
    loader = convnext_loader_warm if epoch < convnext_cfg['warmup_epochs'] else convnext_loader_train
    phase = 'WARM' if epoch < convnext_cfg['warmup_epochs'] else 'AUG'

    losses = []
    convnext_optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'convnext ep{epoch:02d} [{phase}]', disable=not is_main_process())
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and convnext_cfg['use_amp'])):
            logits = convnext_model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = convnext_loss_fn(logits, gt)
        loss = loss / convnext_cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite ConvNeXt loss at epoch={epoch} step={step}: {loss.item()}')

        convnext_scaler.scale(loss).backward()
        if (step + 1) % convnext_cfg['grad_accum'] == 0:
            convnext_scaler.unscale_(convnext_optimizer)
            torch.nn.utils.clip_grad_norm_(convnext_core_model.parameters(), max_norm=1.0)
            convnext_scaler.step(convnext_optimizer)
            convnext_scaler.update()
            convnext_optimizer.zero_grad(set_to_none=True)
            update_ema(convnext_ema_model, convnext_model, convnext_cfg['ema_decay'])

        losses.append(loss.item() * convnext_cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % convnext_cfg['grad_accum'] != 0:
        convnext_scaler.unscale_(convnext_optimizer)
        torch.nn.utils.clip_grad_norm_(convnext_core_model.parameters(), max_norm=1.0)
        convnext_scaler.step(convnext_optimizer)
        convnext_scaler.update()
        convnext_optimizer.zero_grad(set_to_none=True)
        update_ema(convnext_ema_model, convnext_model, convnext_cfg['ema_decay'])

    convnext_scheduler.step()
    barrier()

    if is_main_process():
        cleanup_cuda()
        print('start convnext val model @0.5')
        val_iou_05 = evaluate_iou(convnext_core_model, convnext_loader_val, threshold=0.5, desc=f'convnext val model ep{epoch:02d}')

        cleanup_cuda()
        print('start convnext val ema @0.5')
        val_iou_05_ema = evaluate_iou(convnext_ema_model, convnext_loader_val, threshold=0.5, desc=f'convnext val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - convnext_start) / 60
        backbone_grad_enabled = any(p.requires_grad for p in convnext_core_model.backbone.backbone.parameters())

        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'backbone_trainable': bool(backbone_grad_enabled),
            'minutes': float(elapsed),
        }
        print(
            f"convnext ep{epoch:02d} | loss {np.mean(losses):.3f} | "
            f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
            f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
        )

        if val_iou_05 > convnext_best_iou:
            convnext_best_iou = val_iou_05
            torch.save({
                'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
                'model': convnext_core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': convnext_cfg,
            }, CONVNEXT_RUN_DIR / 'best.pt')
            print('  new convnext best model:', val_iou_05)

        if val_iou_05_ema > convnext_best_ema_iou:
            convnext_best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
                'model': convnext_core_model.state_dict(),
                'ema': convnext_ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': convnext_cfg,
            }, CONVNEXT_RUN_DIR / 'ema_best.pt')
            print('  new convnext best ema:', val_iou_05_ema)

        convnext_log.append(row)
        pd.DataFrame(convnext_log).to_csv(CONVNEXT_RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
            'model': convnext_core_model.state_dict(),
            'ema': convnext_ema_model.state_dict(),
            'optimizer': convnext_optimizer.state_dict(),
            'scheduler': convnext_scheduler.state_dict(),
            'scaler': convnext_scaler.state_dict() if convnext_scaler is not None else None,
            'epoch': epoch,
            'best_iou': float(convnext_best_iou),
            'best_ema_iou': float(convnext_best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': convnext_cfg,
        }, CONVNEXT_RUN_DIR / 'last.pt')

    barrier()


### ConvNeXt Notes

- This continuation block assumes all earlier data-prep and utility cells in the notebook have already been executed.
- It trains a new model in `./runs/v8_convnextv2_fcmae_lss_bev_cleaned` without reusing RTMDet weights.
- Threshold sweep and test submissions should still be done in a separate inference notebook after training.
